# 03 — Modelagem

**Tech Challenge — Fase 2 | POSTECH Data Analytics | Grupo 08**

Escopo: divisão treino/teste, validação cruzada, treino e comparação de seis modelos (mais de dois, como exigido).

**Entradas:** artefatos de `02_preprocessamento.ipynb` (`data/processed/`).
**Saídas:** `validacao_12_meses.joblib` e `resultados_modelagem.joblib`, usados por `04_avaliacao.ipynb`.

Modelos: `DummyClassifier` (referência), Regressão Logística, Árvore de Decisão, Random Forest, Hist Gradient Boosting e KNN. A classe positiva (1) é o atraso de 60+ dias; as métricas de interesse são as da classe 1.

> Todas as saídas abaixo foram geradas no Colab original. As células marcadas como "carregar" ou "salvar" são novas e ficam sem saída até a execução.

In [1]:
# Configuração comum a todos os notebooks do projeto
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

SEMENTE = 42  # semente fixa usada em todo o projeto

# Funciona rodando a partir de notebooks/ ou da raiz do repositório.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = RAIZ / "data" / "raw"              # arquivos brutos (não versionados)
PROCESSED = RAIZ / "data" / "processed"  # artefatos intermediários (não versionados)
PROCESSED.mkdir(parents=True, exist_ok=True)

In [2]:
import joblib

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
)

# Artefatos gerados pelo notebook 02_preprocessamento.ipynb
BASE_MODELO = joblib.load(PROCESSED / "base_modelo.joblib")

_prep = joblib.load(PROCESSED / "preprocessamento.joblib")
preprocessador = _prep["preprocessador"]

_janelas = joblib.load(PROCESSED / "janelas.joblib")
JANELAS_CLIENTES = _janelas["JANELAS_CLIENTES"]
CENARIOS_JANELAS = _janelas["CENARIOS_JANELAS"]
elegiveis_6 = _janelas["elegiveis_6"]
elegiveis_12 = _janelas["elegiveis_12"]
ids_comuns = _janelas["ids_comuns"]
X_cadastro = _janelas["X_cadastro"]
grupos_por_id = _janelas["grupos_por_id"]

print(f"Clientes na base de modelagem: {len(BASE_MODELO):,}")
print(f"Classe 1: {int(BASE_MODELO['ALVO'].sum()):,} ({100 * BASE_MODELO['ALVO'].mean():.2f}%)")

Clientes na base de modelagem: 29,029
Classe 1: 608 (2.09%)


## 1. Divisão treino/teste e referência (baseline)

Divisão estratificada 80/20 por cliente e `DummyClassifier`, que serve de piso: um modelo útil precisa superá-lo em precisão, recall e AUC, não em acurácia.

In [3]:
#INÍCIO DOS TESTES COM DUMMY CLASSIFIER

import pandas as pd
from IPython.display import display

from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

# Entradas e alvo (construídos em 02_preprocessamento.ipynb)
X = BASE_MODELO.drop(columns=['ID', 'ALVO'])
y = BASE_MODELO['ALVO']

# 4. Separar treino e teste, preservando a proporção das classes.
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEMENTE
)

if y_treino.nunique() != 2 or y_teste.nunique() != 2:
    raise ValueError('Treino e teste precisam conter as duas classes.')

# 5. Treinar a referência.
# Não exige codificação ou escala: DummyClassifier ignora as entradas.
modelo_dummy = DummyClassifier(
    strategy='prior',
    random_state=SEMENTE
)

modelo_dummy.fit(X_treino, y_treino)

previsoes = modelo_dummy.predict(X_teste)
indice_classe_1 = list(modelo_dummy.classes_).index(1)
probabilidades = modelo_dummy.predict_proba(X_teste)[:, indice_classe_1]

# 6. Avaliar no teste. Classe positiva = atraso no corte escolhido.
RESULTADOS_DUMMY = pd.DataFrame([{
    'MODELO': 'DummyClassifier — prior',
    'ACURACIA': accuracy_score(y_teste, previsoes),
    'ACURACIA_BALANCEADA': balanced_accuracy_score(y_teste, previsoes),
    'PRECISAO_CLASSE_1': precision_score(
        y_teste, previsoes, zero_division=0
    ),
    'RECALL_CLASSE_1': recall_score(
        y_teste, previsoes, zero_division=0
    ),
    'F1_CLASSE_1': f1_score(
        y_teste, previsoes, zero_division=0
    ),
    'AVERAGE_PRECISION': average_precision_score(y_teste, probabilidades),
    'ROC_AUC': roc_auc_score(y_teste, probabilidades)
}])

MATRIZ_CONFUSAO_DUMMY = pd.DataFrame(
    confusion_matrix(y_teste, previsoes, labels=[0, 1]),
    index=['Real: sem atraso no corte', 'Real: com atraso no corte'],
    columns=['Previsto: sem atraso', 'Previsto: com atraso']
)

print(f'Clientes na base do modelo: {len(BASE_MODELO):,}')
print(f'Clientes no treino: {len(y_treino):,}')
print(f'Clientes no teste: {len(y_teste):,}')
print(f'Percentual com atraso no teste: {100 * y_teste.mean():.2f}%')

display(RESULTADOS_DUMMY.round(4))
display(MATRIZ_CONFUSAO_DUMMY)

Clientes na base do modelo: 29,029
Clientes no treino: 23,223
Clientes no teste: 5,806
Percentual com atraso no teste: 2.10%


,MODELO,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
0,DummyClassifier — prior,0.979,0.5,0.0,0.0,0.0,0.021,0.5


,Previsto: sem atraso,Previsto: com atraso
Real: sem atraso no corte,5684,0
Real: com atraso no corte,122,0


## 2. Modelos

Cada modelo é uma `Pipeline` (pré-processamento + estimador) ajustada só no treino. Classes desbalanceadas são tratadas com `class_weight` onde o modelo permite.

In [4]:
#MODELO DE REGRESSÃO LOGÍSTICA
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

# 1. Preparar cópias dos conjuntos já separados.
X_treino_lr = X_treino.copy()
X_teste_lr = X_teste.copy()

colunas_numericas = X_treino_lr.select_dtypes(
    include='number'
).columns.tolist()

colunas_categoricas = [
    coluna for coluna in X_treino_lr.columns
    if coluna not in colunas_numericas
]

# Uniformizar ausências e tipos para o pré-processamento.
for tabela in [X_treino_lr, X_teste_lr]:
    for coluna in colunas_numericas:
        tabela[coluna] = pd.to_numeric(
            tabela[coluna], errors='raise'
        ).astype('float64')

    for coluna in colunas_categoricas:
        valores = tabela[coluna].astype('string')
        tabela[coluna] = (
            valores.astype(object)
            .where(valores.notna(), float('nan'))
        )

# 2. Pré-processamento: definido em 02_preprocessamento.ipynb
# (imputação + padronização + one-hot; ajustado só no treino de cada divisão).
# A variável `preprocessador` foi carregada na primeira célula.

# 3. Regressão Logística.
# balanced dá maior peso à classe menos frequente.
modelo_logistico = Pipeline([
    ('preprocessamento', preprocessador),
    ('modelo', LogisticRegression(
        class_weight='balanced',
        solver='lbfgs',
        max_iter=2000,
        random_state=42
    ))
])

modelo_logistico.fit(X_treino_lr, y_treino)

# 4. Previsões no teste, com limiar de 0,50.
previsoes_lr = modelo_logistico.predict(X_teste_lr)

indice_classe_1 = list(
    modelo_logistico.named_steps['modelo'].classes_
).index(1)

probabilidades_lr = modelo_logistico.predict_proba(
    X_teste_lr
)[:, indice_classe_1]

# 5. Avaliação — classe positiva = 1.
RESULTADOS_LOGISTICA = pd.DataFrame([{
    'MODELO': 'Regressão Logística — balanced',
    'ACURACIA': accuracy_score(y_teste, previsoes_lr),
    'ACURACIA_BALANCEADA': balanced_accuracy_score(
        y_teste, previsoes_lr
    ),
    'PRECISAO_CLASSE_1': precision_score(
        y_teste, previsoes_lr, zero_division=0
    ),
    'RECALL_CLASSE_1': recall_score(
        y_teste, previsoes_lr, zero_division=0
    ),
    'F1_CLASSE_1': f1_score(
        y_teste, previsoes_lr, zero_division=0
    ),
    'AVERAGE_PRECISION': average_precision_score(
        y_teste, probabilidades_lr
    ),
    'ROC_AUC': roc_auc_score(y_teste, probabilidades_lr)
}])

MATRIZ_CONFUSAO_LOGISTICA = pd.DataFrame(
    confusion_matrix(y_teste, previsoes_lr, labels=[0, 1]),
    index=['Real: sem atraso no corte', 'Real: com atraso no corte'],
    columns=['Previsto: sem atraso', 'Previsto: com atraso']
)

display(RESULTADOS_LOGISTICA.round(4))
display(MATRIZ_CONFUSAO_LOGISTICA)

# 6. Comparar com a referência anterior.
COMPARACAO_MODELOS = pd.concat(
    [RESULTADOS_DUMMY, RESULTADOS_LOGISTICA],
    ignore_index=True
)

print('Comparação dos modelos no mesmo conjunto de teste:')
display(COMPARACAO_MODELOS.round(4))

,MODELO,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
0,Regressão Logística — balanced,0.6567,0.552,0.0273,0.4426,0.0514,0.1027,0.5685


,Previsto: sem atraso,Previsto: com atraso
Real: sem atraso no corte,3759,1925
Real: com atraso no corte,68,54


Comparação dos modelos no mesmo conjunto de teste:


,MODELO,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
0,DummyClassifier — prior,0.9790,0.500,0.0000,0.0000,0.0000,0.0210,0.5000
1,Regressão Logística — balanced,0.6567,0.552,0.0273,0.4426,0.0514,0.1027,0.5685


In [5]:
#MODELO ÁRVORE DE DECISÃO

import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

# 1. Criar uma pipeline independente.
# clone reutiliza a configuração do pré-processamento,
# mas os ajustes são aprendidos novamente somente no treino.
modelo_arvore = Pipeline([
    ('preprocessamento', clone(preprocessador)),
    ('modelo', DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=50,
        class_weight='balanced',
        random_state=42
    ))
])

# Parâmetros iniciais; ainda não foram otimizados.
modelo_arvore.fit(X_treino_lr, y_treino)

# 2. Prever no mesmo conjunto de teste.
previsoes_arvore = modelo_arvore.predict(X_teste_lr)

indice_classe_1 = list(
    modelo_arvore.named_steps['modelo'].classes_
).index(1)

probabilidades_arvore = modelo_arvore.predict_proba(
    X_teste_lr
)[:, indice_classe_1]

# 3. Avaliar a classe positiva = 1.
RESULTADOS_ARVORE = pd.DataFrame([{
    'MODELO': 'Árvore de Decisão — balanced',
    'ACURACIA': accuracy_score(y_teste, previsoes_arvore),
    'ACURACIA_BALANCEADA': balanced_accuracy_score(
        y_teste, previsoes_arvore
    ),
    'PRECISAO_CLASSE_1': precision_score(
        y_teste, previsoes_arvore, zero_division=0
    ),
    'RECALL_CLASSE_1': recall_score(
        y_teste, previsoes_arvore, zero_division=0
    ),
    'F1_CLASSE_1': f1_score(
        y_teste, previsoes_arvore, zero_division=0
    ),
    'AVERAGE_PRECISION': average_precision_score(
        y_teste, probabilidades_arvore
    ),
    'ROC_AUC': roc_auc_score(
        y_teste, probabilidades_arvore
    )
}])

MATRIZ_CONFUSAO_ARVORE = pd.DataFrame(
    confusion_matrix(y_teste, previsoes_arvore, labels=[0, 1]),
    index=['Real: sem atraso no corte', 'Real: com atraso no corte'],
    columns=['Previsto: sem atraso', 'Previsto: com atraso']
)

display(RESULTADOS_ARVORE.round(4))
display(MATRIZ_CONFUSAO_ARVORE)

# 4. Comparar os três modelos.
COMPARACAO_MODELOS = pd.concat(
    [
        RESULTADOS_DUMMY,
        RESULTADOS_LOGISTICA,
        RESULTADOS_ARVORE
    ],
    ignore_index=True
)

print('Comparação no mesmo conjunto de teste:')
display(COMPARACAO_MODELOS.round(4))

,MODELO,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
0,Árvore de Decisão — balanced,0.7773,0.5614,0.0327,0.3361,0.0596,0.033,0.5995


,Previsto: sem atraso,Previsto: com atraso
Real: sem atraso no corte,4472,1212
Real: com atraso no corte,81,41


Comparação no mesmo conjunto de teste:


,MODELO,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
0,DummyClassifier — prior,0.9790,0.5000,0.0000,0.0000,0.0000,0.0210,0.5000
1,Regressão Logística — balanced,0.6567,0.5520,0.0273,0.4426,0.0514,0.1027,0.5685
2,Árvore de Decisão — balanced,0.7773,0.5614,0.0327,0.3361,0.0596,0.0330,0.5995


In [6]:
#MODELO RANDOM FOREST

import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

# 1. Pipeline independente, com parâmetros iniciais.
modelo_random_forest = Pipeline([
    ('preprocessamento', clone(preprocessador)),
    ('modelo', RandomForestClassifier(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=10,
        max_features='sqrt',
        class_weight='balanced_subsample',
        n_jobs=-1,
        random_state=42
    ))
])

# Pré-processamento e modelo ajustados somente no treino.
modelo_random_forest.fit(X_treino_lr, y_treino)

# 2. Previsões no mesmo conjunto de teste.
previsoes_rf = modelo_random_forest.predict(X_teste_lr)

indice_classe_1 = list(
    modelo_random_forest.named_steps['modelo'].classes_
).index(1)

probabilidades_rf = modelo_random_forest.predict_proba(
    X_teste_lr
)[:, indice_classe_1]

# 3. Avaliação — classe positiva = 1.
RESULTADOS_RANDOM_FOREST = pd.DataFrame([{
    'MODELO': 'Random Forest — balanced_subsample',
    'ACURACIA': accuracy_score(y_teste, previsoes_rf),
    'ACURACIA_BALANCEADA': balanced_accuracy_score(
        y_teste, previsoes_rf
    ),
    'PRECISAO_CLASSE_1': precision_score(
        y_teste, previsoes_rf, zero_division=0
    ),
    'RECALL_CLASSE_1': recall_score(
        y_teste, previsoes_rf, zero_division=0
    ),
    'F1_CLASSE_1': f1_score(
        y_teste, previsoes_rf, zero_division=0
    ),
    'AVERAGE_PRECISION': average_precision_score(
        y_teste, probabilidades_rf
    ),
    'ROC_AUC': roc_auc_score(y_teste, probabilidades_rf)
}])

MATRIZ_CONFUSAO_RANDOM_FOREST = pd.DataFrame(
    confusion_matrix(y_teste, previsoes_rf, labels=[0, 1]),
    index=['Real: sem atraso no corte', 'Real: com atraso no corte'],
    columns=['Previsto: sem atraso', 'Previsto: com atraso']
)

display(RESULTADOS_RANDOM_FOREST.round(4))
display(MATRIZ_CONFUSAO_RANDOM_FOREST)

# 4. Comparar os quatro modelos.
COMPARACAO_MODELOS = pd.concat(
    [
        RESULTADOS_DUMMY,
        RESULTADOS_LOGISTICA,
        RESULTADOS_ARVORE,
        RESULTADOS_RANDOM_FOREST
    ],
    ignore_index=True
)

print('Comparação no mesmo conjunto de teste:')
display(COMPARACAO_MODELOS.round(4))

,MODELO,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
0,Random Forest — balanced_subsample,0.9277,0.6302,0.1037,0.3197,0.1566,0.1263,0.708


,Previsto: sem atraso,Previsto: com atraso
Real: sem atraso no corte,5347,337
Real: com atraso no corte,83,39


Comparação no mesmo conjunto de teste:


,MODELO,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
0,DummyClassifier — prior,0.9790,0.5000,0.0000,0.0000,0.0000,0.0210,0.5000
1,Regressão Logística — balanced,0.6567,0.5520,0.0273,0.4426,0.0514,0.1027,0.5685
2,Árvore de Decisão — balanced,0.7773,0.5614,0.0327,0.3361,0.0596,0.0330,0.5995
3,Random Forest — balanced_subsample,0.9277,0.6302,0.1037,0.3197,0.1566,0.1263,0.7080


In [7]:
#MODELO HIST GRADIENT BOOSTING

import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

# 1. Clonar o pré-processamento sem alterar os modelos anteriores.
preprocessador_hgb = clone(preprocessador)

# HistGradientBoosting exige entrada densa.
preprocessador_hgb.set_params(
    categoricas__codificacao__sparse_output=False,
    sparse_threshold=0
)

# 2. Criar o modelo com parâmetros iniciais.
modelo_hist_gradient_boosting = Pipeline([
    ('preprocessamento', preprocessador_hgb),
    ('modelo', HistGradientBoostingClassifier(
        learning_rate=0.1,
        max_iter=200,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=1.0,
        class_weight='balanced',
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42
    ))
])

# Ajustar a pipeline somente no conjunto de treino.
modelo_hist_gradient_boosting.fit(X_treino_lr, y_treino)

# 3. Previsões no mesmo conjunto de teste.
previsoes_hgb = modelo_hist_gradient_boosting.predict(X_teste_lr)

indice_classe_1 = list(
    modelo_hist_gradient_boosting.named_steps['modelo'].classes_
).index(1)

probabilidades_hgb = (
    modelo_hist_gradient_boosting
    .predict_proba(X_teste_lr)[:, indice_classe_1]
)

# 4. Avaliação — classe positiva = 1.
RESULTADOS_HIST_GRADIENT_BOOSTING = pd.DataFrame([{
    'MODELO': 'Hist Gradient Boosting — balanced',
    'ACURACIA': accuracy_score(y_teste, previsoes_hgb),
    'ACURACIA_BALANCEADA': balanced_accuracy_score(
        y_teste, previsoes_hgb
    ),
    'PRECISAO_CLASSE_1': precision_score(
        y_teste, previsoes_hgb, zero_division=0
    ),
    'RECALL_CLASSE_1': recall_score(
        y_teste, previsoes_hgb, zero_division=0
    ),
    'F1_CLASSE_1': f1_score(
        y_teste, previsoes_hgb, zero_division=0
    ),
    'AVERAGE_PRECISION': average_precision_score(
        y_teste, probabilidades_hgb
    ),
    'ROC_AUC': roc_auc_score(
        y_teste, probabilidades_hgb
    )
}])

MATRIZ_CONFUSAO_HGB = pd.DataFrame(
    confusion_matrix(y_teste, previsoes_hgb, labels=[0, 1]),
    index=['Real: sem atraso no corte', 'Real: com atraso no corte'],
    columns=['Previsto: sem atraso', 'Previsto: com atraso']
)

print(
    'Iterações realizadas: '
    f"{modelo_hist_gradient_boosting.named_steps['modelo'].n_iter_}"
)

display(RESULTADOS_HIST_GRADIENT_BOOSTING.round(4))
display(MATRIZ_CONFUSAO_HGB)

# 5. Comparar os cinco modelos.
COMPARACAO_MODELOS = pd.concat(
    [
        RESULTADOS_DUMMY,
        RESULTADOS_LOGISTICA,
        RESULTADOS_ARVORE,
        RESULTADOS_RANDOM_FOREST,
        RESULTADOS_HIST_GRADIENT_BOOSTING
    ],
    ignore_index=True
)

print('Comparação no mesmo conjunto de teste:')
display(COMPARACAO_MODELOS.round(4))

Iterações realizadas: 68


,MODELO,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
0,Hist Gradient Boosting — balanced,0.8782,0.6811,0.0827,0.4754,0.1409,0.1848,0.7058


,Previsto: sem atraso,Previsto: com atraso
Real: sem atraso no corte,5041,643
Real: com atraso no corte,64,58


Comparação no mesmo conjunto de teste:


,MODELO,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
0,DummyClassifier — prior,0.9790,0.5000,0.0000,0.0000,0.0000,0.0210,0.5000
1,Regressão Logística — balanced,0.6567,0.5520,0.0273,0.4426,0.0514,0.1027,0.5685
2,Árvore de Decisão — balanced,0.7773,0.5614,0.0327,0.3361,0.0596,0.0330,0.5995
3,Random Forest — balanced_subsample,0.9277,0.6302,0.1037,0.3197,0.1566,0.1263,0.7080
4,Hist Gradient Boosting — balanced,0.8782,0.6811,0.0827,0.4754,0.1409,0.1848,0.7058


In [8]:
#MODELO KNN CLASSIFIER

import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

# 1. Criar uma pipeline independente.
modelo_knn = Pipeline([
    ('preprocessamento', clone(preprocessador)),
    ('modelo', KNeighborsClassifier(
        n_neighbors=15,
        weights='distance',
        metric='euclidean',
        algorithm='brute',
        n_jobs=-1
    ))
])

# Pré-processamento ajustado somente no treino.
modelo_knn.fit(X_treino_lr, y_treino)

# 2. Previsões no mesmo conjunto de teste.
previsoes_knn = modelo_knn.predict(X_teste_lr)

indice_classe_1 = list(
    modelo_knn.named_steps['modelo'].classes_
).index(1)

probabilidades_knn = modelo_knn.predict_proba(
    X_teste_lr
)[:, indice_classe_1]

# 3. Avaliação — classe positiva = 1.
RESULTADOS_KNN = pd.DataFrame([{
    'MODELO': 'KNN — 15 vizinhos, distance',
    'ACURACIA': accuracy_score(y_teste, previsoes_knn),
    'ACURACIA_BALANCEADA': balanced_accuracy_score(
        y_teste, previsoes_knn
    ),
    'PRECISAO_CLASSE_1': precision_score(
        y_teste, previsoes_knn, zero_division=0
    ),
    'RECALL_CLASSE_1': recall_score(
        y_teste, previsoes_knn, zero_division=0
    ),
    'F1_CLASSE_1': f1_score(
        y_teste, previsoes_knn, zero_division=0
    ),
    'AVERAGE_PRECISION': average_precision_score(
        y_teste, probabilidades_knn
    ),
    'ROC_AUC': roc_auc_score(
        y_teste, probabilidades_knn
    )
}])

MATRIZ_CONFUSAO_KNN = pd.DataFrame(
    confusion_matrix(y_teste, previsoes_knn, labels=[0, 1]),
    index=['Real: sem atraso no corte', 'Real: com atraso no corte'],
    columns=['Previsto: sem atraso', 'Previsto: com atraso']
)

display(RESULTADOS_KNN.round(4))
display(MATRIZ_CONFUSAO_KNN)

# 4. Comparar os seis modelos.
COMPARACAO_MODELOS = pd.concat(
    [
        RESULTADOS_DUMMY,
        RESULTADOS_LOGISTICA,
        RESULTADOS_ARVORE,
        RESULTADOS_RANDOM_FOREST,
        RESULTADOS_HIST_GRADIENT_BOOSTING,
        RESULTADOS_KNN
    ],
    ignore_index=True
)

print('Comparação no mesmo conjunto de teste:')
display(COMPARACAO_MODELOS.round(4))

,MODELO,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
0,"KNN — 15 vizinhos, distance",0.9786,0.576,0.475,0.1557,0.2346,0.2257,0.7383


,Previsto: sem atraso,Previsto: com atraso
Real: sem atraso no corte,5663,21
Real: com atraso no corte,103,19


Comparação no mesmo conjunto de teste:


,MODELO,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
0,DummyClassifier — prior,0.9790,0.5000,0.0000,0.0000,0.0000,0.0210,0.5000
1,Regressão Logística — balanced,0.6567,0.5520,0.0273,0.4426,0.0514,0.1027,0.5685
2,Árvore de Decisão — balanced,0.7773,0.5614,0.0327,0.3361,0.0596,0.0330,0.5995
3,Random Forest — balanced_subsample,0.9277,0.6302,0.1037,0.3197,0.1566,0.1263,0.7080
4,Hist Gradient Boosting — balanced,0.8782,0.6811,0.0827,0.4754,0.1409,0.1848,0.7058
5,"KNN — 15 vizinhos, distance",0.9786,0.5760,0.4750,0.1557,0.2346,0.2257,0.7383


## 3. Vazamento por perfis repetidos e nova divisão

87,2% do teste tinha um cadastro idêntico a algum do treino, o que infla o desempenho de modelos que "memorizam" (KNN e Random Forest). A divisão é refeita agrupando perfis idênticos, de modo que cada perfil fique inteiro no treino ou no teste, e todos os modelos são treinados de novo.

In [9]:
#IDENTIFICA PERFIS COM VALORES IGUAIS EM TODAS AS VARIAVEIS CADASTRAIS USADAS PELO MODELO

import pandas as pd
from IPython.display import display

# 1. Conferir as variáveis comparadas.
colunas_perfil = X_treino.columns.tolist()

if set(colunas_perfil) != set(X_teste.columns):
    raise ValueError('Treino e teste possuem colunas diferentes.')

if 'ID' in colunas_perfil or 'ALVO' in colunas_perfil:
    raise ValueError('Retire ID e ALVO das variáveis comparadas.')

if not colunas_perfil:
    raise ValueError('Não há variáveis cadastrais para comparar.')

# 2. Reunir cópias dos cadastros antes do pré-processamento.
perfis = pd.concat(
    [
        X_treino[colunas_perfil],
        X_teste[colunas_perfil]
    ],
    ignore_index=True
)

# Agrupar por igualdade de todas as colunas.
# dropna=False considera ausências na comparação.
grupos = (
    perfis.groupby(
        colunas_perfil,
        dropna=False,
        observed=True,
        sort=False
    )
    .ngroup()
)

# Acrescentar informações apenas à tabela de diagnóstico.
perfis['_PERFIL'] = grupos
perfis['_ORIGEM'] = (
    ['TREINO'] * len(X_treino)
    + ['TESTE'] * len(X_teste)
)
perfis['_INDICE_BASE'] = (
    X_treino.index.tolist()
    + X_teste.index.tolist()
)

# 3. Contar ocorrências de cada perfil nos dois conjuntos.
RESUMO_PERFIS = (
    perfis.groupby(['_PERFIL', '_ORIGEM'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['TREINO', 'TESTE'], fill_value=0)
)

PERFIS_COMPARTILHADOS = RESUMO_PERFIS.loc[
    RESUMO_PERFIS['TREINO'].gt(0)
    & RESUMO_PERFIS['TESTE'].gt(0)
].copy()

# 4. Recuperar todos os cadastros com perfis compartilhados.
CADASTROS_IGUAIS_TREINO_TESTE = (
    perfis.loc[
        perfis['_PERFIL'].isin(PERFIS_COMPARTILHADOS.index)
    ]
    .sort_values(['_PERFIL', '_ORIGEM'])
    .reset_index(drop=True)
)

n_treino = int(PERFIS_COMPARTILHADOS['TREINO'].sum())
n_teste = int(PERFIS_COMPARTILHADOS['TESTE'].sum())

percentual_teste = (
    100 * n_teste / len(X_teste)
    if len(X_teste) else 0
)

print(f'Perfis distintos no conjunto completo: {len(RESUMO_PERFIS):,}')
print(
    'Perfis presentes simultaneamente em treino e teste: '
    f'{len(PERFIS_COMPARTILHADOS):,}'
)
print(f'Linhas do treino com esses perfis: {n_treino:,}')
print(f'Linhas do teste com esses perfis: {n_teste:,}')
print(
    'Percentual do teste com cadastro igual a algum cadastro do treino: '
    f'{percentual_teste:.2f}%'
)

print('\nContagem por perfil compartilhado — primeiros 20:')
display(PERFIS_COMPARTILHADOS.head(20))

print('\nCadastros compartilhados — primeiras 20 linhas:')

Perfis distintos no conjunto completo: 8,744
Perfis presentes simultaneamente em treino e teste: 3,222
Linhas do treino com esses perfis: 12,589
Linhas do teste com esses perfis: 5,063
Percentual do teste com cadastro igual a algum cadastro do treino: 87.20%

Contagem por perfil compartilhado — primeiros 20:


_ORIGEM,TREINO,TESTE
_PERFIL,,
1,2,1
3,2,1
4,8,1
6,5,1
8,8,1
9,8,2
12,11,3
13,7,1
18,10,3



Cadastros compartilhados — primeiras 20 linhas:


In [10]:
#CRIA UM GRUPO POR PERFIL CADASTRAL
#REFAZ A SEPARAÇÃO MANTENDO CADA GRUPO INTEIRAMENTE NO TREINO OU TESTE
#TREINA NOVAMENTE TODOS OS MODELOS

import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix
)

# 1. Preservar a comparação inicial.
# Não sobrescrever essa cópia ao executar a célula novamente.
if 'COMPARACAO_INICIAL' not in globals():
    COMPARACAO_INICIAL = COMPARACAO_MODELOS.copy()

# Usar exatamente os clientes e o alvo já definidos.
base_agrupada = BASE_MODELO.copy().reset_index(drop=True)

if base_agrupada['ID'].isna().any():
    raise ValueError('Existem IDs ausentes.')

if base_agrupada['ID'].duplicated().any():
    raise ValueError('A base deve conter uma única linha por ID.')

X_agrupado = base_agrupada.drop(columns=['ID', 'ALVO']).copy()
y_agrupado = base_agrupada['ALVO'].copy()

if y_agrupado.isna().any() or set(y_agrupado.unique()) != {0, 1}:
    raise ValueError('O alvo deve conter as classes 0 e 1, sem ausências.')

# 2. Criar grupos por igualdade de todas as variáveis cadastrais.
# ID e ALVO ficam fora da definição do perfil.
# Ausências nas mesmas posições são consideradas iguais.
GRUPOS_PERFIL = (
    X_agrupado.groupby(
        X_agrupado.columns.tolist(),
        dropna=False,
        observed=True,
        sort=False
    )
    .ngroup()
)

if GRUPOS_PERFIL.nunique() < 5:
    raise ValueError('São necessários pelo menos 5 perfis distintos.')

# 3. Nova divisão. Usar a primeira divisão, sem escolher por desempenho.
divisor = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

indices_treino, indices_teste = next(
    divisor.split(X_agrupado, y_agrupado, groups=GRUPOS_PERFIL)
)

X_treino_grupos = X_agrupado.iloc[indices_treino].copy()
X_teste_grupos = X_agrupado.iloc[indices_teste].copy()

y_treino_grupos = y_agrupado.iloc[indices_treino].copy()
y_teste_grupos = y_agrupado.iloc[indices_teste].copy()

grupos_treino = set(GRUPOS_PERFIL.iloc[indices_treino])
grupos_teste = set(GRUPOS_PERFIL.iloc[indices_teste])

sobreposicao = grupos_treino & grupos_teste

if sobreposicao:
    raise ValueError('Há perfis compartilhados entre treino e teste.')

if (
    y_treino_grupos.nunique() != 2
    or y_teste_grupos.nunique() != 2
):
    raise ValueError(
        'A divisão agrupada não contém ambas as classes nos dois conjuntos.'
    )

# Guardar quais IDs pertencem a cada conjunto.
DIVISAO_POR_PERFIL = base_agrupada[['ID', 'ALVO']].copy()
DIVISAO_POR_PERFIL['GRUPO_PERFIL'] = GRUPOS_PERFIL
DIVISAO_POR_PERFIL['CONJUNTO'] = 'TREINO'
DIVISAO_POR_PERFIL.loc[indices_teste, 'CONJUNTO'] = 'TESTE'

print(f'Perfis distintos: {GRUPOS_PERFIL.nunique():,}')
print(f'Perfis compartilhados: {len(sobreposicao)}')
print(f'Clientes no treino: {len(y_treino_grupos):,}')
print(f'Clientes no teste: {len(y_teste_grupos):,}')
print(f'Percentual no teste: {100 * len(indices_teste) / len(base_agrupada):.2f}%')

DISTRIBUICAO_CLASSES_GRUPOS = pd.DataFrame([
    {
        'CONJUNTO': nome,
        'CLASSE_0': int(alvo.eq(0).sum()),
        'CLASSE_1': int(alvo.eq(1).sum()),
        'PERCENTUAL_CLASSE_1': round(100 * alvo.mean(), 2)
    }
    for nome, alvo in [
        ('TREINO', y_treino_grupos),
        ('TESTE', y_teste_grupos)
    ]
])

display(DISTRIBUICAO_CLASSES_GRUPOS)

# 4. Uniformizar tipos, sem aprender estatísticas do teste.
# Imputação, escala e codificação serão ajustadas dentro das pipelines.
numericas = X_treino_grupos.select_dtypes(
    include='number'
).columns.tolist()

categoricas = [
    c for c in X_treino_grupos.columns if c not in numericas
]

for tabela in [X_treino_grupos, X_teste_grupos]:
    for coluna in numericas:
        tabela[coluna] = pd.to_numeric(
            tabela[coluna], errors='raise'
        ).astype('float64')

    for coluna in categoricas:
        valores = tabela[coluna].astype('string')
        tabela[coluna] = (
            valores.astype(object)
            .where(valores.notna(), float('nan'))
        )

# 5. Clonar os modelos anteriores.
# clone mantém a configuração, mas remove os ajustes já aprendidos.
MODELOS_POR_PERFIL = {
    'DummyClassifier — prior': clone(modelo_dummy),
    'Regressão Logística — balanced': clone(modelo_logistico),
    'Árvore de Decisão — balanced': clone(modelo_arvore),
    'Random Forest — balanced_subsample': clone(modelo_random_forest),
    'Hist Gradient Boosting — balanced': clone(
        modelo_hist_gradient_boosting
    ),
    'KNN — 15 vizinhos, distance': clone(modelo_knn)
}

# Desativar a divisão interna aleatória do HGB, que poderia
# compartilhar perfis no treino/validação da parada antecipada.
# O modelo passa a executar o máximo configurado de 200 iterações.
MODELOS_POR_PERFIL[
    'Hist Gradient Boosting — balanced'
].set_params(modelo__early_stopping=False)

# 6. Treinar e avaliar os seis modelos na mesma divisão agrupada.
resultados = []
MATRIZES_CONFUSAO_POR_PERFIL = {}

for nome, modelo in MODELOS_POR_PERFIL.items():
    print(f'\nTreinando: {nome}')

    modelo.fit(X_treino_grupos, y_treino_grupos)

    previsoes = modelo.predict(X_teste_grupos)
    indice_classe_1 = list(modelo.classes_).index(1)
    probabilidades = modelo.predict_proba(
        X_teste_grupos
    )[:, indice_classe_1]

    resultados.append({
        'MODELO': nome,
        'ACURACIA': accuracy_score(y_teste_grupos, previsoes),
        'ACURACIA_BALANCEADA': balanced_accuracy_score(
            y_teste_grupos, previsoes
        ),
        'PRECISAO_CLASSE_1': precision_score(
            y_teste_grupos, previsoes, zero_division=0
        ),
        'RECALL_CLASSE_1': recall_score(
            y_teste_grupos, previsoes, zero_division=0
        ),
        'F1_CLASSE_1': f1_score(
            y_teste_grupos, previsoes, zero_division=0
        ),
        'AVERAGE_PRECISION': average_precision_score(
            y_teste_grupos, probabilidades
        ),
        'ROC_AUC': roc_auc_score(
            y_teste_grupos, probabilidades
        )
    })

    MATRIZES_CONFUSAO_POR_PERFIL[nome] = pd.DataFrame(
        confusion_matrix(
            y_teste_grupos, previsoes, labels=[0, 1]
        ),
        index=['Real: classe 0', 'Real: classe 1'],
        columns=['Previsto: classe 0', 'Previsto: classe 1']
    )

COMPARACAO_POR_PERFIL = pd.DataFrame(resultados)

print('\nRESULTADOS COM PERFIS SEPARADOS:')
display(COMPARACAO_POR_PERFIL.round(4))

# 7. Mostrar os dois protocolos separadamente.
COMPARACAO_PROTOCOLOS = pd.concat(
    [
        COMPARACAO_INICIAL.assign(
            PROTOCOLO='Divisão inicial por ID'
        ),
        COMPARACAO_POR_PERFIL.assign(
            PROTOCOLO='Divisão por perfil cadastral'
        )
    ],
    ignore_index=True
)

print('\nCOMPARAÇÃO DOS PROTOCOLOS:')
display(
    COMPARACAO_PROTOCOLOS[
        ['PROTOCOLO', 'MODELO', 'PRECISAO_CLASSE_1',
         'RECALL_CLASSE_1', 'F1_CLASSE_1',
         'AVERAGE_PRECISION', 'ROC_AUC']
    ].round(4)
)

display(
    MATRIZES_CONFUSAO_POR_PERFIL['KNN — 15 vizinhos, distance']
)

Perfis distintos: 8,744
Perfis compartilhados: 0
Clientes no treino: 23,224
Clientes no teste: 5,805
Percentual no teste: 20.00%


,CONJUNTO,CLASSE_0,CLASSE_1,PERCENTUAL_CLASSE_1
0,TREINO,22737,487,2.10
1,TESTE,5684,121,2.08



Treinando: DummyClassifier — prior

Treinando: Regressão Logística — balanced

Treinando: Árvore de Decisão — balanced

Treinando: Random Forest — balanced_subsample

Treinando: Hist Gradient Boosting — balanced

Treinando: KNN — 15 vizinhos, distance

RESULTADOS COM PERFIS SEPARADOS:


,MODELO,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
0,DummyClassifier — prior,0.9792,0.5000,0.0000,0.0000,0.0000,0.0208,0.5000
1,Regressão Logística — balanced,0.6331,0.5457,0.0260,0.4545,0.0491,0.0773,0.5612
2,Árvore de Decisão — balanced,0.6059,0.5359,0.0246,0.4628,0.0467,0.0269,0.5587
3,Random Forest — balanced_subsample,0.9214,0.5393,0.0461,0.1405,0.0694,0.0451,0.5296
4,Hist Gradient Boosting — balanced,0.9320,0.5810,0.0798,0.2149,0.1163,0.1022,0.5402
5,"KNN — 15 vizinhos, distance",0.9783,0.4996,0.0000,0.0000,0.0000,0.0261,0.5358



COMPARAÇÃO DOS PROTOCOLOS:


,PROTOCOLO,MODELO,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
0,Divisão inicial por ID,DummyClassifier — prior,0.0000,0.0000,0.0000,0.0210,0.5000
1,Divisão inicial por ID,Regressão Logística — balanced,0.0273,0.4426,0.0514,0.1027,0.5685
2,Divisão inicial por ID,Árvore de Decisão — balanced,0.0327,0.3361,0.0596,0.0330,0.5995
3,Divisão inicial por ID,Random Forest — balanced_subsample,0.1037,0.3197,0.1566,0.1263,0.7080
4,Divisão inicial por ID,Hist Gradient Boosting — balanced,0.0827,0.4754,0.1409,0.1848,0.7058
5,Divisão inicial por ID,"KNN — 15 vizinhos, distance",0.4750,0.1557,0.2346,0.2257,0.7383
6,Divisão por perfil cadastral,DummyClassifier — prior,0.0000,0.0000,0.0000,0.0208,0.5000
7,Divisão por perfil cadastral,Regressão Logística — balanced,0.0260,0.4545,0.0491,0.0773,0.5612
8,Divisão por perfil cadastral,Árvore de Decisão — balanced,0.0246,0.4628,0.0467,0.0269,0.5587
9,Divisão por perfil cadastral,Random Forest — balanced_subsample,0.0461,0.1405,0.0694,0.0451,0.5296


,Previsto: classe 0,Previsto: classe 1
Real: classe 0,5679,5
Real: classe 1,121,0


## 4. Validação cruzada de 5 folds por perfil

`StratifiedGroupKFold` com os mesmos folds para todos os modelos. Média e desvio padrão das métricas da classe 1.

In [11]:
#EXECUTA CINCO FOLDS POR PERFIL PARA AVALIAR O MELHOR MODELO
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score
)

# 1. Recuperar dados e grupos da etapa anterior.
X_cv = X_agrupado.copy().reset_index(drop=True)
y_cv = y_agrupado.copy().reset_index(drop=True)
grupos_cv = GRUPOS_PERFIL.copy().reset_index(drop=True)

if not (len(X_cv) == len(y_cv) == len(grupos_cv)):
    raise ValueError('Dados, alvo e grupos possuem tamanhos diferentes.')

if grupos_cv.isna().any():
    raise ValueError('Existem grupos ausentes.')

if y_cv.isna().any() or set(y_cv.unique()) != {0, 1}:
    raise ValueError('O alvo deve conter as classes 0 e 1.')

# Uniformizar tipos sem calcular estatísticas da base completa.
numericas = X_cv.select_dtypes(include='number').columns.tolist()
categoricas = [c for c in X_cv.columns if c not in numericas]

for coluna in numericas:
    X_cv[coluna] = pd.to_numeric(
        X_cv[coluna], errors='raise'
    ).astype('float64')

for coluna in categoricas:
    valores = X_cv[coluna].astype('string')
    X_cv[coluna] = (
        valores.astype(object)
        .where(valores.notna(), float('nan'))
    )

# 2. Definir os mesmos cinco folds para todos os modelos.
divisor_cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

folds = list(divisor_cv.split(X_cv, y_cv, groups=grupos_cv))

# Validar todas as divisões antes dos treinamentos.
diagnostico_folds = []

for numero, (idx_treino, idx_teste) in enumerate(folds, start=1):
    compartilhados = (
        set(grupos_cv.iloc[idx_treino])
        & set(grupos_cv.iloc[idx_teste])
    )

    if compartilhados:
        raise ValueError(f'Fold {numero}: há perfis compartilhados.')

    if (
        y_cv.iloc[idx_treino].nunique() != 2
        or y_cv.iloc[idx_teste].nunique() != 2
    ):
        raise ValueError(f'Fold {numero}: faltam exemplos de alguma classe.')

    diagnostico_folds.append({
        'FOLD': numero,
        'CLIENTES_TREINO': len(idx_treino),
        'CLIENTES_TESTE': len(idx_teste),
        'CLASSE_1_TREINO': int(y_cv.iloc[idx_treino].sum()),
        'CLASSE_1_TESTE': int(y_cv.iloc[idx_teste].sum()),
        'PERCENTUAL_CLASSE_1_TESTE':
            100 * y_cv.iloc[idx_teste].mean(),
        'PERFIS_COMPARTILHADOS': len(compartilhados)
    })

DIAGNOSTICO_FOLDS = pd.DataFrame(diagnostico_folds)
display(DIAGNOSTICO_FOLDS.round(2))

# 3. Clonar configurações dos seis modelos da etapa anterior.
# Os modelos já treinados não serão alterados.
configuracoes_cv = {
    nome: clone(modelo)
    for nome, modelo in MODELOS_POR_PERFIL.items()
}

# Manter HGB sem divisão interna aleatória para parada antecipada.
for modelo in configuracoes_cv.values():
    if 'modelo__early_stopping' in modelo.get_params():
        modelo.set_params(modelo__early_stopping=False)

# 4. Treinar e avaliar em cada fold.
resultados_cv = []

for numero, (idx_treino, idx_teste) in enumerate(folds, start=1):
    X_tr = X_cv.iloc[idx_treino]
    X_te = X_cv.iloc[idx_teste]
    y_tr = y_cv.iloc[idx_treino]
    y_te = y_cv.iloc[idx_teste]

    for nome, configuracao in configuracoes_cv.items():
        print(f'Fold {numero}/5 — {nome}', flush=True)

        # Nova instância em cada fold:
        # imputação, escala e codificação aprendidas somente em X_tr.
        modelo = clone(configuracao)
        modelo.fit(X_tr, y_tr)

        previsoes = modelo.predict(X_te)
        indice_classe_1 = list(modelo.classes_).index(1)
        probabilidades = modelo.predict_proba(
            X_te
        )[:, indice_classe_1]

        resultados_cv.append({
            'MODELO': nome,
            'FOLD': numero,
            'ACURACIA': accuracy_score(y_te, previsoes),
            'ACURACIA_BALANCEADA': balanced_accuracy_score(
                y_te, previsoes
            ),
            'PRECISAO_CLASSE_1': precision_score(
                y_te, previsoes, zero_division=0
            ),
            'RECALL_CLASSE_1': recall_score(
                y_te, previsoes, zero_division=0
            ),
            'F1_CLASSE_1': f1_score(
                y_te, previsoes, zero_division=0
            ),
            'AVERAGE_PRECISION': average_precision_score(
                y_te, probabilidades
            ),
            'ROC_AUC': roc_auc_score(y_te, probabilidades)
        })

RESULTADOS_CV_POR_FOLD = pd.DataFrame(resultados_cv)

# 5. Resumir média, desvio padrão, mínimo e máximo.
metricas = [
    'ACURACIA',
    'ACURACIA_BALANCEADA',
    'PRECISAO_CLASSE_1',
    'RECALL_CLASSE_1',
    'F1_CLASSE_1',
    'AVERAGE_PRECISION',
    'ROC_AUC'
]

resultados_longos = RESULTADOS_CV_POR_FOLD.melt(
    id_vars=['MODELO', 'FOLD'],
    value_vars=metricas,
    var_name='METRICA',
    value_name='VALOR'
)

RESUMO_CV = (
    resultados_longos.groupby(['MODELO', 'METRICA'], sort=False)
    .agg(
        MEDIA=('VALOR', 'mean'),
        DESVIO_PADRAO=('VALOR', 'std'),
        MINIMO=('VALOR', 'min'),
        MAXIMO=('VALOR', 'max')
    )
    .reset_index()
)

MEDIAS_CV = (
    RESULTADOS_CV_POR_FOLD.groupby('MODELO')[metricas]
    .mean()
    .sort_values('AVERAGE_PRECISION', ascending=False)
)

DESVIOS_CV = (
    RESULTADOS_CV_POR_FOLD.groupby('MODELO')[metricas]
    .std()
    .reindex(MEDIAS_CV.index)
)

print('\nMÉDIAS DOS CINCO FOLDS:')
display(MEDIAS_CV.round(4))

print('\nDESVIOS PADRÃO DOS CINCO FOLDS:')
display(DESVIOS_CV.round(4))

print('\nDETALHES DAS MÉTRICAS DA CLASSE 1:')
display(
    RESUMO_CV.loc[
        RESUMO_CV['METRICA'].isin([
            'PRECISAO_CLASSE_1',
            'RECALL_CLASSE_1',
            'F1_CLASSE_1',
            'AVERAGE_PRECISION'
        ])
    ].round(4)
)

,FOLD,CLIENTES_TREINO,CLIENTES_TESTE,CLASSE_1_TREINO,CLASSE_1_TESTE,PERCENTUAL_CLASSE_1_TESTE,PERFIS_COMPARTILHADOS
0,1,23224,5805,487,121,2.08,0
1,2,23224,5805,487,121,2.08,0
2,3,23222,5807,486,122,2.10,0
3,4,23223,5806,486,122,2.10,0
4,5,23223,5806,486,122,2.10,0


Fold 1/5 — DummyClassifier — prior
Fold 1/5 — Regressão Logística — balanced
Fold 1/5 — Árvore de Decisão — balanced
Fold 1/5 — Random Forest — balanced_subsample
Fold 1/5 — Hist Gradient Boosting — balanced
Fold 1/5 — KNN — 15 vizinhos, distance
Fold 2/5 — DummyClassifier — prior
Fold 2/5 — Regressão Logística — balanced
Fold 2/5 — Árvore de Decisão — balanced
Fold 2/5 — Random Forest — balanced_subsample
Fold 2/5 — Hist Gradient Boosting — balanced
Fold 2/5 — KNN — 15 vizinhos, distance
Fold 3/5 — DummyClassifier — prior
Fold 3/5 — Regressão Logística — balanced
Fold 3/5 — Árvore de Decisão — balanced
Fold 3/5 — Random Forest — balanced_subsample
Fold 3/5 — Hist Gradient Boosting — balanced
Fold 3/5 — KNN — 15 vizinhos, distance
Fold 4/5 — DummyClassifier — prior
Fold 4/5 — Regressão Logística — balanced
Fold 4/5 — Árvore de Decisão — balanced
Fold 4/5 — Random Forest — balanced_subsample
Fold 4/5 — Hist Gradient Boosting — balanced
Fold 4/5 — KNN — 15 vizinhos, distance
Fold 5/5 — D

,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
MODELO,,,,,,,
Hist Gradient Boosting — balanced,0.9417,0.5663,0.0826,0.1744,0.1111,0.0917,0.5850
Regressão Logística — balanced,0.6232,0.5339,0.0248,0.4408,0.0469,0.0670,0.5377
Random Forest — balanced_subsample,0.9362,0.5352,0.0518,0.1168,0.0713,0.0376,0.5572
"KNN — 15 vizinhos, distance",0.9784,0.4997,0.0000,0.0000,0.0000,0.0259,0.5327
Árvore de Decisão — balanced,0.6488,0.5365,0.0255,0.4194,0.0479,0.0247,0.5474
DummyClassifier — prior,0.9791,0.5000,0.0000,0.0000,0.0000,0.0209,0.5000



DESVIOS PADRÃO DOS CINCO FOLDS:


,ACURACIA,ACURACIA_BALANCEADA,PRECISAO_CLASSE_1,RECALL_CLASSE_1,F1_CLASSE_1,AVERAGE_PRECISION,ROC_AUC
MODELO,,,,,,,
Hist Gradient Boosting — balanced,0.0082,0.0115,0.0080,0.0308,0.0087,0.0151,0.0308
Regressão Logística — balanced,0.0218,0.0243,0.0029,0.0378,0.0053,0.0177,0.0286
Random Forest — balanced_subsample,0.0097,0.0157,0.0146,0.0341,0.0195,0.0092,0.0307
"KNN — 15 vizinhos, distance",0.0006,0.0003,0.0000,0.0000,0.0000,0.0018,0.0185
Árvore de Decisão — balanced,0.0878,0.0025,0.0012,0.0942,0.0016,0.0022,0.0222
DummyClassifier — prior,0.0001,0.0000,0.0000,0.0000,0.0000,0.0001,0.0000



DETALHES DAS MÉTRICAS DA CLASSE 1:


,MODELO,METRICA,MEDIA,DESVIO_PADRAO,MINIMO,MAXIMO
12,DummyClassifier — prior,PRECISAO_CLASSE_1,0.0000,0.0000,0.0000,0.0000
13,Regressão Logística — balanced,PRECISAO_CLASSE_1,0.0248,0.0029,0.0219,0.0292
14,Árvore de Decisão — balanced,PRECISAO_CLASSE_1,0.0255,0.0012,0.0245,0.0275
15,Random Forest — balanced_subsample,PRECISAO_CLASSE_1,0.0518,0.0146,0.0411,0.0775
16,Hist Gradient Boosting — balanced,PRECISAO_CLASSE_1,0.0826,0.0080,0.0746,0.0955
17,"KNN — 15 vizinhos, distance",PRECISAO_CLASSE_1,0.0000,0.0000,0.0000,0.0000
18,DummyClassifier — prior,RECALL_CLASSE_1,0.0000,0.0000,0.0000,0.0000
19,Regressão Logística — balanced,RECALL_CLASSE_1,0.4408,0.0378,0.4016,0.4918
20,Árvore de Decisão — balanced,RECALL_CLASSE_1,0.4194,0.0942,0.3115,0.5574
21,Random Forest — balanced_subsample,RECALL_CLASSE_1,0.1168,0.0341,0.0820,0.1639


## 5. Janelas de 6 e 12 meses (Dummy e Hist Gradient Boosting)

Avalia o melhor modelo nos quatro cenários de janela montados no notebook 02.

In [12]:
# Folds por perfil para os cenários de janela (continuação da seção 4)
def criar_folds(ids, alvo):
    grupos_cenario = grupos_por_id.loc[ids]

    if grupos_cenario.nunique() < 5:
        raise ValueError('Há menos de cinco perfis neste cenário.')

    divisor = StratifiedGroupKFold(
        n_splits=5, shuffle=True, random_state=SEMENTE
    )

    return list(divisor.split(
        X_cadastro.loc[ids],
        alvo,
        groups=grupos_cenario
    ))

# Mesmos folds nas duas janelas da coorte comum.
# A estratificação usa o alvo de 12 meses.
folds_comuns = criar_folds(
    ids_comuns,
    elegiveis_12.loc[ids_comuns, 'ALVO_JANELA'].astype(int)
)

# ============================================================
# 5. Avaliar Dummy e HGB em cinco folds por cenário
# ============================================================

configuracoes = {
    'Dummy': clone(MODELOS_POR_PERFIL['DummyClassifier — prior']),
    'Hist Gradient Boosting': clone(
        MODELOS_POR_PERFIL['Hist Gradient Boosting — balanced']
    ).set_params(modelo__early_stopping=False)
}

resultados = []
diagnostico_folds = []

for cenario, tabela in CENARIOS_JANELAS.items():
    ids = tabela.index
    X_cenario = X_cadastro.loc[ids]
    y_cenario = tabela['ALVO_JANELA'].astype(int)
    grupos_cenario = grupos_por_id.loc[ids]

    folds_cenario = (
        folds_comuns
        if 'coorte comum' in cenario
        else criar_folds(ids, y_cenario)
    )

    for numero, (idx_tr, idx_te) in enumerate(folds_cenario, start=1):
        X_tr, X_te = X_cenario.iloc[idx_tr], X_cenario.iloc[idx_te]
        y_tr, y_te = y_cenario.iloc[idx_tr], y_cenario.iloc[idx_te]

        compartilhados = (
            set(grupos_cenario.iloc[idx_tr])
            & set(grupos_cenario.iloc[idx_te])
        )

        if compartilhados:
            raise ValueError('Há perfis compartilhados.')

        if y_tr.nunique() != 2 or y_te.nunique() != 2:
            raise ValueError(
                f'{cenario}, fold {numero}: faltam exemplos de uma classe. '
                'Não é possível avaliar este cenário com os cinco folds atuais.'
            )

        diagnostico_folds.append({
            'CENARIO': cenario,
            'FOLD': numero,
            'CLIENTES_TESTE': len(y_te),
            'CLASSE_1_TESTE': int(y_te.sum()),
            'PERFIS_COMPARTILHADOS': len(compartilhados)
        })

        for nome, configuracao in configuracoes.items():
            print(f'{cenario} | fold {numero}/5 | {nome}', flush=True)

            modelo = clone(configuracao)
            modelo.fit(X_tr, y_tr)

            previsto = modelo.predict(X_te)
            indice_1 = list(modelo.classes_).index(1)
            probabilidades = modelo.predict_proba(X_te)[:, indice_1]

            resultados.append({
                'CENARIO': cenario,
                'FOLD': numero,
                'MODELO': nome,
                'ACURACIA': accuracy_score(y_te, previsto),
                'ACURACIA_BALANCEADA':
                    balanced_accuracy_score(y_te, previsto),
                'PRECISAO_CLASSE_1':
                    precision_score(y_te, previsto, zero_division=0),
                'RECALL_CLASSE_1':
                    recall_score(y_te, previsto, zero_division=0),
                'F1_CLASSE_1':
                    f1_score(y_te, previsto, zero_division=0),
                'AVERAGE_PRECISION':
                    average_precision_score(y_te, probabilidades),
                'ROC_AUC':
                    roc_auc_score(y_te, probabilidades)
            })

RESULTADOS_JANELAS_FOLDS = pd.DataFrame(resultados)
DIAGNOSTICO_FOLDS_JANELAS = pd.DataFrame(diagnostico_folds)

metricas = [
    'ACURACIA', 'ACURACIA_BALANCEADA',
    'PRECISAO_CLASSE_1', 'RECALL_CLASSE_1',
    'F1_CLASSE_1', 'AVERAGE_PRECISION', 'ROC_AUC'
]

MEDIAS_JANELAS = (
    RESULTADOS_JANELAS_FOLDS.groupby(['CENARIO', 'MODELO'])[metricas]
    .mean()
)

DESVIOS_JANELAS = (
    RESULTADOS_JANELAS_FOLDS.groupby(['CENARIO', 'MODELO'])[metricas]
    .std()
)

print('\nMÉDIAS DOS CINCO FOLDS:')
display(MEDIAS_JANELAS.round(4))

print('\nDESVIOS PADRÃO DOS CINCO FOLDS:')
display(DESVIOS_JANELAS.round(4))

6 meses — elegíveis | fold 1/5 | Dummy
6 meses — elegíveis | fold 1/5 | Hist Gradient Boosting
6 meses — elegíveis | fold 2/5 | Dummy
6 meses — elegíveis | fold 2/5 | Hist Gradient Boosting
6 meses — elegíveis | fold 3/5 | Dummy
6 meses — elegíveis | fold 3/5 | Hist Gradient Boosting
6 meses — elegíveis | fold 4/5 | Dummy
6 meses — elegíveis | fold 4/5 | Hist Gradient Boosting
6 meses — elegíveis | fold 5/5 | Dummy
6 meses — elegíveis | fold 5/5 | Hist Gradient Boosting
12 meses — elegíveis | fold 1/5 | Dummy
12 meses — elegíveis | fold 1/5 | Hist Gradient Boosting
12 meses — elegíveis | fold 2/5 | Dummy
12 meses — elegíveis | fold 2/5 | Hist Gradient Boosting
12 meses — elegíveis | fold 3/5 | Dummy
12 meses — elegíveis | fold 3/5 | Hist Gradient Boosting
12 meses — elegíveis | fold 4/5 | Dummy
12 meses — elegíveis | fold 4/5 | Hist Gradient Boosting
12 meses — elegíveis | fold 5/5 | Dummy
12 meses — elegíveis | fold 5/5 | Hist Gradient Boosting
6 meses — coorte comum | fold 1/5 | Dumm

ACURACIA  ACURACIA_BALANCEADA  \
CENARIO                 MODELO                                                  
12 meses — coorte comum Dummy                     0.9829               0.5000   
                        Hist Gradient Boosting    0.9632               0.5932   
12 meses — elegíveis    Dummy                     0.9830               0.5000   
                        Hist Gradient Boosting    0.9642               0.5729   
6 meses — coorte comum  Dummy                     0.9927               0.5000   
                        Hist Gradient Boosting    0.9856               0.5841   
6 meses — elegíveis     Dummy                     0.9931               0.5000   
                        Hist Gradient Boosting    0.9857               0.6191   

                                                PRECISAO_CLASSE_1  \
CENARIO                 MODELO                                      
12 meses — coorte comum Dummy                              0.0000   
                        Hist Gradient Boosting             0.1313   
12 meses — elegíveis    Dummy                              0.0000   
                        Hist Gradient Boosting             0.1170   
6 meses — coorte comum  Dummy                              0.0000   
                        Hist Gradient Boosting             0.1487   
6 meses — elegíveis     Dummy                              0.0000   
                        Hist Gradient Boosting             0.2101   

                                                RECALL_CLASSE_1  F1_CLASSE_1  \
CENARIO                 MODELO                                                 
12 meses — coorte comum Dummy                            0.0000       0.0000   
                        Hist Gradient Boosting           0.2100       0.1607   
12 meses — elegíveis    Dummy                            0.0000       0.0000   
                        Hist Gradient Boosting           0.1678       0.1375   
6 meses — coorte comum  Dummy                            0.0000       0.0000   
                        Hist Gradient Boosting           0.1767       0.1564   
6 meses — elegíveis     Dummy                            0.0000       0.0000   
                        Hist Gradient Boosting           0.2474       0.2112   

                                                AVERAGE_PRECISION  ROC_AUC  
CENARIO                 MODELO                                              
12 meses — coorte comum Dummy                              0.0171   0.5000  
                        Hist Gradient Boosting             0.1254   0.6194  
12 meses — elegíveis    Dummy                              0.0170   0.5000  
                        Hist Gradient Boosting             0.1096   0.6204  
6 meses — coorte comum  Dummy                              0.0073   0.5000  
                        Hist Gradient Boosting             0.1087   0.5885  
6 meses — elegíveis     Dummy                              0.0069   0.5000  
                        Hist Gradient Boosting             0.1473   0.6359


DESVIOS PADRÃO DOS CINCO FOLDS:


ACURACIA  ACURACIA_BALANCEADA  \
CENARIO                 MODELO                                                  
12 meses — coorte comum Dummy                     0.0002               0.0000   
                        Hist Gradient Boosting    0.0035               0.0326   
12 meses — elegíveis    Dummy                     0.0002               0.0000   
                        Hist Gradient Boosting    0.0026               0.0224   
6 meses — coorte comum  Dummy                     0.0013               0.0000   
                        Hist Gradient Boosting    0.0029               0.0392   
6 meses — elegíveis     Dummy                     0.0000               0.0000   
                        Hist Gradient Boosting    0.0057               0.0311   

                                                PRECISAO_CLASSE_1  \
CENARIO                 MODELO                                      
12 meses — coorte comum Dummy                              0.0000   
                        Hist Gradient Boosting             0.0339   
12 meses — elegíveis    Dummy                              0.0000   
                        Hist Gradient Boosting             0.0307   
6 meses — coorte comum  Dummy                              0.0000   
                        Hist Gradient Boosting             0.0843   
6 meses — elegíveis     Dummy                              0.0000   
                        Hist Gradient Boosting             0.1397   

                                                RECALL_CLASSE_1  F1_CLASSE_1  \
CENARIO                 MODELO                                                 
12 meses — coorte comum Dummy                            0.0000       0.0000   
                        Hist Gradient Boosting           0.0674       0.0457   
12 meses — elegíveis    Dummy                            0.0000       0.0000   
                        Hist Gradient Boosting           0.0443       0.0353   
6 meses — coorte comum  Dummy                            0.0000       0.0000   
                        Hist Gradient Boosting           0.0769       0.0742   
6 meses — elegíveis     Dummy                            0.0000       0.0000   
                        Hist Gradient Boosting           0.0606       0.0853   

                                                AVERAGE_PRECISION  ROC_AUC  
CENARIO                 MODELO                                              
12 meses — coorte comum Dummy                              0.0002   0.0000  
                        Hist Gradient Boosting             0.0401   0.0318  
12 meses — elegíveis    Dummy                              0.0002   0.0000  
                        Hist Gradient Boosting             0.0456   0.0445  
6 meses — coorte comum  Dummy                              0.0013   0.0000  
                        Hist Gradient Boosting             0.0402   0.0425  
6 meses — elegíveis     Dummy                              0.0000   0.0000  
                        Hist Gradient Boosting             0.0526   0.0743

## 6. Base e folds do modelo de 12 meses

Alinha por ID os clientes elegíveis da janela de 12 meses (`X_limiar`, `y_limiar`, perfis) e define os 5 folds externos e 3 internos, todos separados por perfil. Os mesmos folds são usados na busca de hiperparâmetros (seção 7) e, no notebook 04, no ajuste do limiar e na importância das variáveis.

In [13]:
# VALIDAÇÃO ALINHADA POR ID PARA O MODELO DE 12 MESES (folds externos e internos por perfil)

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    confusion_matrix
)

# ============================================================
# 1. Configuração e objetos do experimento anterior
# ============================================================

SEMENTE = 42
FOLDS_EXTERNOS = 5
FOLDS_INTERNOS = 3
LIMIAR_PADRAO = 0.50

objetos_necessarios = [
    "JANELAS_CLIENTES",
    "X_cadastro",
    "grupos_por_id",
    "MODELOS_POR_PERFIL"
]

faltantes = [
    nome for nome in objetos_necessarios
    if nome not in globals()
]

if faltantes:
    raise ValueError(
        "Execute primeiro a célula das janelas de 6 e 12 meses. "
        f"Objetos ausentes: {faltantes}"
    )

nome_modelo = "Hist Gradient Boosting — balanced"

if nome_modelo not in MODELOS_POR_PERFIL:
    raise ValueError(
        f"O modelo '{nome_modelo}' não foi encontrado "
        "em MODELOS_POR_PERFIL."
    )

# ============================================================
# Preparar a base alinhando explicitamente os IDs
# ============================================================

janela_12 = JANELAS_CLIENTES[12].copy()

# O resumo pode ter ID como coluna ou como índice.
if "ID" in janela_12.columns:
    janela_12 = janela_12.set_index("ID")
elif janela_12.index.name != "ID":
    raise ValueError(
        "JANELAS_CLIENTES[12] precisa possuir uma coluna ID "
        "ou um índice chamado ID. Não use o índice das linhas "
        "como identificador dos clientes."
    )

X_por_id = X_cadastro.copy()

if "ID" in X_por_id.columns:
    X_por_id = X_por_id.set_index("ID")
elif X_por_id.index.name != "ID":
    raise ValueError(
        "X_cadastro precisa estar indexado pelo ID do cliente."
    )

grupos_por_id_local = grupos_por_id.copy()

if not isinstance(grupos_por_id_local, pd.Series):
    raise ValueError(
        "grupos_por_id precisa ser uma Series indexada "
        "pelos IDs dos clientes."
    )


def normalizar_indice_id(objeto, nome):
    objeto = objeto.copy()

    indice = pd.Index(
        pd.array(objeto.index, dtype="string"),
        name="ID"
    ).str.strip()

    if indice.isna().any() or (indice == "").any():
        raise ValueError(f"{nome}: existem IDs ausentes ou vazios.")

    if indice.has_duplicates:
        raise ValueError(f"{nome}: existem IDs duplicados.")

    objeto.index = indice
    return objeto


# Padroniza o tipo dos IDs nas três estruturas.
janela_12 = normalizar_indice_id(
    janela_12, "JANELAS_CLIENTES[12]"
)

X_por_id = normalizar_indice_id(
    X_por_id, "X_cadastro"
)

grupos_por_id_local = normalizar_indice_id(
    grupos_por_id_local, "grupos_por_id"
)

# Seleciona os IDs reais dos clientes elegíveis.
ids = janela_12.index[
    janela_12["ELEGIVEL"].fillna(False).astype(bool)
]

if len(ids) == 0:
    raise ValueError("Não existem clientes elegíveis para 12 meses.")

# Valida correspondência antes de selecionar.
sem_cadastro = ids.difference(X_por_id.index)
sem_perfil = ids.difference(grupos_por_id_local.index)

if len(sem_cadastro) or len(sem_perfil):
    raise ValueError(
        f"Clientes elegíveis sem cadastro: {len(sem_cadastro)}. "
        f"Clientes elegíveis sem perfil: {len(sem_perfil)}. "
        "Reexecute a célula das janelas para reconstruir "
        "os objetos com o mesmo universo de clientes."
    )

if janela_12.loc[ids, "ALVO_JANELA"].isna().any():
    raise ValueError("Existem clientes elegíveis sem alvo.")

# .loc[ids] garante a mesma ordem em X, y e grupos.
X_limiar = (
    X_por_id.loc[ids]
    .copy()
    .reset_index(drop=True)
)

y_limiar = (
    janela_12.loc[ids, "ALVO_JANELA"]
    .astype(int)
    .reset_index(drop=True)
)

grupos_limiar = (
    grupos_por_id_local.loc[ids]
    .reset_index(drop=True)
)

ids_limiar = pd.Series(ids.to_numpy(), name="ID")

if set(y_limiar.unique()) != {0, 1}:
    raise ValueError("A base precisa conter as classes 0 e 1.")

if grupos_limiar.isna().any():
    raise ValueError("Existem clientes sem grupo de perfil.")

print("Alinhamento por ID concluído.")
print(f"Clientes elegíveis: {len(ids):,}")
print(f"Classe 1: {int(y_limiar.sum()):,}")

# Clona o pipeline completo: pré-processamento + modelo.
# Cada ajuste aprende imputação e codificação somente no treino.
modelo_base = clone(MODELOS_POR_PERFIL[nome_modelo])

# Evita validação interna aleatória do HGB.
modelo_base.set_params(modelo__early_stopping=False)

print(f"Clientes elegíveis: {len(y_limiar):,}")
print(f"Classe 1: {int(y_limiar.sum()):,}")
print(f"Percentual classe 1: {100 * y_limiar.mean():.2f}%")
print(f"Perfis distintos: {grupos_limiar.nunique():,}")


# ============================================================
# 2. Funções auxiliares
# ============================================================

def validar_divisao(y, grupos, treino, teste, descricao):
    """Valida classes e ausência de perfis compartilhados."""

    compartilhados = (
        set(grupos.iloc[treino])
        & set(grupos.iloc[teste])
    )

    if compartilhados:
        raise ValueError(
            f"{descricao}: existem perfis compartilhados."
        )

    if (
        y.iloc[treino].nunique() != 2
        or y.iloc[teste].nunique() != 2
    ):
        raise ValueError(
            f"{descricao}: treino e validação/teste "
            "precisam conter as duas classes."
        )


def probabilidades_classe_1(modelo, X):
    indice = list(modelo.classes_).index(1)
    return modelo.predict_proba(X)[:, indice]


def escolher_limiar_f1(y, probabilidades):
    """
    Escolhe o limiar com maior F1 nas previsões internas
    fora da amostra. Em empate, escolhe o maior limiar.
    """

    precisao, recall, limiares = precision_recall_curve(
        y, probabilidades
    )

    # O último ponto da curva não possui limiar correspondente.
    precisao = precisao[:-1]
    recall = recall[:-1]

    f1 = np.divide(
        2 * precisao * recall,
        precisao + recall,
        out=np.zeros_like(precisao),
        where=(precisao + recall) > 0
    )

    melhor_f1 = f1.max()

    empatados = np.flatnonzero(
        np.isclose(f1, melhor_f1, rtol=0, atol=1e-12)
    )

    indice = empatados[-1]

    return {
        "LIMIAR_AJUSTADO": float(limiares[indice]),
        "F1_INTERNO": float(f1[indice]),
        "PRECISAO_INTERNA": float(precisao[indice]),
        "RECALL_INTERNO": float(recall[indice])
    }


def avaliar_limiar(y, probabilidades, limiar):
    previsoes = (probabilidades >= limiar).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y, previsoes, labels=[0, 1]
    ).ravel()

    return {
        "ACURACIA": accuracy_score(y, previsoes),
        "ACURACIA_BALANCEADA": balanced_accuracy_score(
            y, previsoes
        ),
        "PRECISAO_CLASSE_1": precision_score(
            y, previsoes, zero_division=0
        ),
        "RECALL_CLASSE_1": recall_score(
            y, previsoes, zero_division=0
        ),
        "F1_CLASSE_1": f1_score(
            y, previsoes, zero_division=0
        ),
        "AVERAGE_PRECISION": average_precision_score(
            y, probabilidades
        ),
        "ROC_AUC": roc_auc_score(y, probabilidades),
        "VERDADEIROS_NEGATIVOS": int(tn),
        "FALSOS_POSITIVOS": int(fp),
        "FALSOS_NEGATIVOS": int(fn),
        "VERDADEIROS_POSITIVOS": int(tp),
        "CLIENTES_SINALIZADOS": int(fp + tp)
    }


# ============================================================
# 3. Preparar e validar todas as divisões antes de treinar
# ============================================================

cv_externo = StratifiedGroupKFold(
    n_splits=FOLDS_EXTERNOS,
    shuffle=True,
    random_state=SEMENTE
)

divisoes_externas = list(
    cv_externo.split(
        X_limiar, y_limiar, groups=grupos_limiar
    )
)

plano_folds = []
diagnosticos_internos = []

for fold, (treino, teste) in enumerate(
    divisoes_externas, start=1
):
    validar_divisao(
        y_limiar, grupos_limiar,
        treino, teste,
        f"Fold externo {fold}"
    )

    X_treino = X_limiar.iloc[treino].reset_index(drop=True)
    y_treino = y_limiar.iloc[treino].reset_index(drop=True)
    grupos_treino = (
        grupos_limiar.iloc[treino].reset_index(drop=True)
    )

    cv_interno = StratifiedGroupKFold(
        n_splits=FOLDS_INTERNOS,
        shuffle=True,
        random_state=SEMENTE + fold
    )

    divisoes_internas = list(
        cv_interno.split(
            X_treino, y_treino, groups=grupos_treino
        )
    )

    for fold_interno, (tr_int, val_int) in enumerate(
        divisoes_internas, start=1
    ):
        validar_divisao(
            y_treino, grupos_treino,
            tr_int, val_int,
            f"Fold externo {fold}, interno {fold_interno}"
        )

        diagnosticos_internos.append({
            "FOLD_EXTERNO": fold,
            "FOLD_INTERNO": fold_interno,
            "CLIENTES_TREINO": len(tr_int),
            "CLIENTES_VALIDACAO": len(val_int),
            "CLASSE_1_TREINO": int(
                y_treino.iloc[tr_int].sum()
            ),
            "CLASSE_1_VALIDACAO": int(
                y_treino.iloc[val_int].sum()
            ),
            "PERFIS_COMPARTILHADOS": 0
        })

    plano_folds.append(
        (fold, treino, teste, divisoes_internas)
    )

Alinhamento por ID concluído.
Clientes elegíveis: 21,731
Classe 1: 370
Clientes elegíveis: 21,731
Classe 1: 370
Percentual classe 1: 1.70%
Perfis distintos: 7,473


## 7. Busca de hiperparâmetros (Hist Gradient Boosting)

Compara o modelo atual com 8 configurações (`max_leaf_nodes`, `min_samples_leaf`, `l2_regularization`) em validação aninhada, selecionando por Average Precision. A busca **não melhorou** o modelo (F1 médio de 0,146 para 0,078), então o notebook 04 mantém os parâmetros originais.

In [14]:
#COMPARA O MODELO ATUAL COM OITO CONFIGURAÇÕES

import numpy as np
import pandas as pd

from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV

# ============================================================
# 1. Objetos e configuração
# ============================================================

necessarios = [
    "X_limiar",
    "y_limiar",
    "grupos_limiar",
    "ids_limiar",
    "modelo_base",
    "plano_folds",
    "validar_divisao",
    "probabilidades_classe_1",
    "avaliar_limiar"
]

faltantes = [
    nome for nome in necessarios
    if nome not in globals()
]

if faltantes:
    raise ValueError(
        "Execute primeiro as células anteriores. "
        f"Objetos ausentes: {faltantes}"
    )

LIMIAR = 0.50

# Mantém os demais parâmetros do modelo atual.
# 2 × 2 × 2 = 8 configurações.
GRADE_PARAMETROS = {
    "modelo__max_leaf_nodes": [7, 15],
    "modelo__min_samples_leaf": [30, 100],
    "modelo__l2_regularization": [1.0, 10.0]
}

modelo_referencia = clone(modelo_base)
modelo_referencia.set_params(
    modelo__early_stopping=False
)

print(f"Clientes: {len(y_limiar):,}")
print(f"Classe 1: {int(y_limiar.sum()):,}")
print(f"Perfis: {grupos_limiar.nunique():,}")
print("Critério de seleção interna: Average Precision")
print("Limiar de avaliação externa: 0.50")
print("Configurações por busca: 8")


# ============================================================
# 2. Busca interna e avaliação externa
# ============================================================

registros_resultados = []
registros_parametros = []
registros_buscas = []
registros_previsoes = []

for fold, treino, teste, divisoes_internas in plano_folds:
    print(f"\n{'=' * 45}")
    print(f"Fold externo {fold}/{len(plano_folds)}")

    validar_divisao(
        y_limiar,
        grupos_limiar,
        treino,
        teste,
        f"Fold externo {fold}"
    )

    X_treino = X_limiar.iloc[treino].reset_index(drop=True)
    y_treino = y_limiar.iloc[treino].reset_index(drop=True)
    grupos_treino = (
        grupos_limiar.iloc[treino].reset_index(drop=True)
    )

    X_teste = X_limiar.iloc[teste]
    y_teste = y_limiar.iloc[teste]

    # Os índices internos se referem ao treino externo.
    # São as mesmas divisões utilizadas nas células anteriores.
    for interno, (tr_int, val_int) in enumerate(
        divisoes_internas, start=1
    ):
        validar_divisao(
            y_treino,
            grupos_treino,
            tr_int,
            val_int,
            f"Fold externo {fold}, interno {interno}"
        )

    # O pipeline completo é ajustado em cada treino interno.
    # Assim, imputação e codificação não usam a validação.
    busca = GridSearchCV(
        estimator=clone(modelo_referencia),
        param_grid=GRADE_PARAMETROS,
        scoring="average_precision",
        cv=divisoes_internas,
        refit=True,
        n_jobs=1,
        return_train_score=False,
        error_score="raise",
        verbose=2
    )

    print("Buscando parâmetros no treino externo...")
    busca.fit(X_treino, y_treino)

    # best_estimator_ já foi reajustado em todo o treino externo.
    modelo_otimizado = busca.best_estimator_

    print(f"Melhores parâmetros: {busca.best_params_}")
    print(
        "Average Precision média interna: "
        f"{busca.best_score_:.4f}"
    )

    registros_parametros.append({
        "FOLD": fold,
        "CLIENTES_TREINO": len(treino),
        "CLIENTES_TESTE": len(teste),
        "CLASSE_1_TREINO": int(y_treino.sum()),
        "CLASSE_1_TESTE": int(y_teste.sum()),
        "PERFIS_COMPARTILHADOS": 0,
        "AVERAGE_PRECISION_INTERNA": busca.best_score_,
        "MAX_LEAF_NODES": (
            busca.best_params_["modelo__max_leaf_nodes"]
        ),
        "MIN_SAMPLES_LEAF": (
            busca.best_params_["modelo__min_samples_leaf"]
        ),
        "L2_REGULARIZATION": (
            busca.best_params_["modelo__l2_regularization"]
        )
    })

    # Guarda todas as configurações testadas.
    resultados_busca = pd.DataFrame(busca.cv_results_)

    colunas_busca = [
        "params",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]

    tabela_busca = resultados_busca[colunas_busca].copy()
    tabela_busca.insert(0, "FOLD_EXTERNO", fold)
    tabela_busca = tabela_busca.rename(columns={
        "params": "PARAMETROS",
        "mean_test_score": "AP_INTERNA_MEDIA",
        "std_test_score": "AP_INTERNA_DESVIO_PADRAO",
        "rank_test_score": "POSICAO_INTERNA"
    })

    registros_buscas.append(tabela_busca)

    # Treina a referência no mesmo treino externo.
    print("Treinando o modelo de referência...")
    modelo_atual = clone(modelo_referencia)
    modelo_atual.fit(X_treino, y_treino)

    modelos_avaliados = {
        "HGB atual": modelo_atual,
        "HGB otimizado": modelo_otimizado
    }

    # Somente agora utiliza o teste externo.
    for nome, modelo in modelos_avaliados.items():
        probabilidades = probabilidades_classe_1(
            modelo, X_teste
        )

        avaliacao = avaliar_limiar(
            y_teste, probabilidades, LIMIAR
        )

        registros_resultados.append({
            "FOLD": fold,
            "MODELO": nome,
            "CLIENTES_TESTE": len(teste),
            "CLASSE_1_TESTE": int(y_teste.sum()),
            "LIMIAR": LIMIAR,
            **avaliacao
        })

        registros_previsoes.append(pd.DataFrame({
            "ID": ids_limiar.iloc[teste].to_numpy(),
            "GRUPO_PERFIL": (
                grupos_limiar.iloc[teste].to_numpy()
            ),
            "FOLD": fold,
            "MODELO": nome,
            "ALVO": y_teste.to_numpy(),
            "PROBABILIDADE_CLASSE_1": probabilidades,
            "PREVISAO": (
                probabilidades >= LIMIAR
            ).astype(int)
        }))

        print(
            f"{nome}: "
            f"AP externa = "
            f"{avaliacao['AVERAGE_PRECISION']:.4f}; "
            f"ROC AUC = {avaliacao['ROC_AUC']:.4f}"
        )


# ============================================================
# 3. Consolidar os resultados
# ============================================================

RESULTADOS_BUSCA_HGB_FOLDS = pd.DataFrame(
    registros_resultados
)

PARAMETROS_HGB_POR_FOLD = pd.DataFrame(
    registros_parametros
)

DETALHES_BUSCAS_HGB = pd.concat(
    registros_buscas, ignore_index=True
)

PREVISOES_EXTERNAS_BUSCA_HGB = pd.concat(
    registros_previsoes, ignore_index=True
)

metricas = [
    "ACURACIA",
    "ACURACIA_BALANCEADA",
    "PRECISAO_CLASSE_1",
    "RECALL_CLASSE_1",
    "F1_CLASSE_1",
    "AVERAGE_PRECISION",
    "ROC_AUC"
]

ordem = ["HGB atual", "HGB otimizado"]

MEDIAS_BUSCA_HGB = (
    RESULTADOS_BUSCA_HGB_FOLDS
    .groupby("MODELO")[metricas]
    .mean()
    .reindex(ordem)
)

DESVIOS_BUSCA_HGB = (
    RESULTADOS_BUSCA_HGB_FOLDS
    .groupby("MODELO")[metricas]
    .std()
    .reindex(ordem)
)

# Comparação pareada no mesmo fold.
atual = (
    RESULTADOS_BUSCA_HGB_FOLDS
    .loc[
        RESULTADOS_BUSCA_HGB_FOLDS["MODELO"]
        .eq("HGB atual")
    ]
    .set_index("FOLD")[metricas]
)

otimizado = (
    RESULTADOS_BUSCA_HGB_FOLDS
    .loc[
        RESULTADOS_BUSCA_HGB_FOLDS["MODELO"]
        .eq("HGB otimizado")
    ]
    .set_index("FOLD")[metricas]
)

DIFERENCAS_BUSCA_HGB_FOLDS = otimizado.subtract(atual)

RESUMO_DIFERENCAS_BUSCA_HGB = (
    DIFERENCAS_BUSCA_HGB_FOLDS
    .agg(["mean", "std", "min", "max"])
    .T
    .rename(columns={
        "mean": "DIFERENCA_MEDIA",
        "std": "DESVIO_PADRAO",
        "min": "MINIMO",
        "max": "MAXIMO"
    })
)

RESUMO_DIFERENCAS_BUSCA_HGB["FOLDS_COM_MELHORA"] = (
    DIFERENCAS_BUSCA_HGB_FOLDS.gt(0).sum()
)

# Frequência das configurações selecionadas.
FREQUENCIA_PARAMETROS_HGB = (
    PARAMETROS_HGB_POR_FOLD
    .groupby([
        "MAX_LEAF_NODES",
        "MIN_SAMPLES_LEAF",
        "L2_REGULARIZATION"
    ])
    .size()
    .reset_index(name="FOLDS_SELECIONADOS")
    .sort_values(
        "FOLDS_SELECIONADOS", ascending=False
    )
)

colunas_contagem = [
    "VERDADEIROS_NEGATIVOS",
    "FALSOS_POSITIVOS",
    "FALSOS_NEGATIVOS",
    "VERDADEIROS_POSITIVOS",
    "CLIENTES_SINALIZADOS"
]

CONTAGENS_BUSCA_HGB = (
    RESULTADOS_BUSCA_HGB_FOLDS
    .groupby("MODELO")[colunas_contagem]
    .sum()
    .reindex(ordem)
)


# ============================================================
# 4. Apresentar resultados
# ============================================================

print("\nMELHORES PARÂMETROS EM CADA FOLD:")
display(PARAMETROS_HGB_POR_FOLD.round(4))

print("\nFREQUÊNCIA DAS CONFIGURAÇÕES SELECIONADAS:")
display(FREQUENCIA_PARAMETROS_HGB)

print("\nRESULTADOS DOS TESTES EXTERNOS POR FOLD:")
display(RESULTADOS_BUSCA_HGB_FOLDS.round(4))

print("\nMÉDIAS DOS CINCO TESTES EXTERNOS:")
display(MEDIAS_BUSCA_HGB.round(4))

print("\nDESVIOS PADRÃO DOS CINCO TESTES EXTERNOS:")
display(DESVIOS_BUSCA_HGB.round(4))

print("\nDIFERENÇAS POR FOLD — OTIMIZADO MENOS ATUAL:")
display(DIFERENCAS_BUSCA_HGB_FOLDS.round(4))

print("\nRESUMO DAS DIFERENÇAS:")
display(RESUMO_DIFERENCAS_BUSCA_HGB.round(4))

print("\nCONTAGENS SOMADAS DOS TESTES EXTERNOS:")
display(CONTAGENS_BUSCA_HGB)

for nome in ordem:
    linha = CONTAGENS_BUSCA_HGB.loc[nome]

    matriz = pd.DataFrame(
        [
            [
                int(linha["VERDADEIROS_NEGATIVOS"]),
                int(linha["FALSOS_POSITIVOS"])
            ],
            [
                int(linha["FALSOS_NEGATIVOS"]),
                int(linha["VERDADEIROS_POSITIVOS"])
            ]
        ],
        index=["Real: classe 0", "Real: classe 1"],
        columns=["Previsto: classe 0", "Previsto: classe 1"]
    )

    print(f"\nMATRIZ DE CONFUSÃO — {nome}:")
    display(matriz)

Clientes: 21,731
Classe 1: 370
Perfis: 7,473
Critério de seleção interna: Average Precision
Limiar de avaliação externa: 0.50
Configurações por busca: 8

Fold externo 1/5
Buscando parâmetros no treino externo...
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END modelo__l2_regularization=1.0, modelo__max_leaf_nodes=7, modelo__min_samples_leaf=30; total time=   0.7s
[CV] END modelo__l2_regularization=1.0, modelo__max_leaf_nodes=7, modelo__min_samples_leaf=30; total time=   0.7s
[CV] END modelo__l2_regularization=1.0, modelo__max_leaf_nodes=7, modelo__min_samples_leaf=30; total time=   0.7s
[CV] END modelo__l2_regularization=1.0, modelo__max_leaf_nodes=7, modelo__min_samples_leaf=100; total time=   0.7s
[CV] END modelo__l2_regularization=1.0, modelo__max_leaf_nodes=7, modelo__min_samples_leaf=100; total time=   0.7s
[CV] END modelo__l2_regularization=1.0, modelo__max_leaf_nodes=7, modelo__min_samples_leaf=100; total time=   0.7s
[CV] END modelo__l2_regularization=1.0, m

,FOLD,CLIENTES_TREINO,CLIENTES_TESTE,CLASSE_1_TREINO,CLASSE_1_TESTE,PERFIS_COMPARTILHADOS,AVERAGE_PRECISION_INTERNA,MAX_LEAF_NODES,MIN_SAMPLES_LEAF,L2_REGULARIZATION
0,1,17386,4345,296,74,0,0.0969,15,30,1.0
1,2,17383,4348,295,75,0,0.0596,15,100,1.0
2,3,17385,4346,296,74,0,0.0759,15,30,1.0
3,4,17383,4348,296,74,0,0.0633,15,100,10.0
4,5,17387,4344,297,73,0,0.0600,15,30,1.0



FREQUÊNCIA DAS CONFIGURAÇÕES SELECIONADAS:


,MAX_LEAF_NODES,MIN_SAMPLES_LEAF,L2_REGULARIZATION,FOLDS_SELECIONADOS
0,15,30,1.0,3
1,15,100,1.0,1
2,15,100,10.0,1



RESULTADOS DOS TESTES EXTERNOS POR FOLD:


...

,VERDADEIROS_NEGATIVOS,FALSOS_POSITIVOS,FALSOS_NEGATIVOS,VERDADEIROS_POSITIVOS,CLIENTES_SINALIZADOS
MODELO,,,,,
HGB atual,20890,471,308,62,533
HGB otimizado,19629,1732,277,93,1825



MATRIZ DE CONFUSÃO — HGB atual:


,Previsto: classe 0,Previsto: classe 1
Real: classe 0,20890,471
Real: classe 1,308,62



MATRIZ DE CONFUSÃO — HGB otimizado:


,Previsto: classe 0,Previsto: classe 1
Real: classe 0,19629,1732
Real: classe 1,277,93


## 8. Salvar artefatos

In [15]:
# Salvar os objetos da validação de 12 meses e os resultados para o notebook 04.
joblib.dump(
    {
        "X_limiar": X_limiar,
        "y_limiar": y_limiar,
        "grupos_limiar": grupos_limiar,
        "ids_limiar": ids_limiar,
        "plano_folds": plano_folds,
        "diagnosticos_internos": diagnosticos_internos,
        "modelo_base": modelo_base,
        "MODELOS_POR_PERFIL": MODELOS_POR_PERFIL,
    },
    PROCESSED / "validacao_12_meses.joblib",
)

joblib.dump(
    {
        "COMPARACAO_PROTOCOLOS": COMPARACAO_PROTOCOLOS,
        "MEDIAS_CV": MEDIAS_CV,
        "DESVIOS_CV": DESVIOS_CV,
        "MEDIAS_JANELAS": MEDIAS_JANELAS,
        "DESVIOS_JANELAS": DESVIOS_JANELAS,
        "MEDIAS_BUSCA_HGB": MEDIAS_BUSCA_HGB,
        "RESULTADOS_BUSCA_HGB_FOLDS": RESULTADOS_BUSCA_HGB_FOLDS,
    },
    PROCESSED / "resultados_modelagem.joblib",
)

print("Artefatos salvos em:", PROCESSED)

Artefatos salvos em: d:\GitHub\Tech-Challenge-Fase-2-POSTECH-Data-Analytics\data\processed


**Conclusão da modelagem (resultados do Colab):** com perfis separados, o Hist Gradient Boosting é o melhor modelo na validação cruzada (ROC-AUC 0,59, F1 0,12, precisão 0,09); KNN e Random Forest só pareciam bons por causa do vazamento. O modelo segue para `04_avaliacao.ipynb`.